# N09 — Tire Degradation TCN: Global Model

**Goal:** Train a causal TCN to predict per-lap tire degradation from race telemetry.

The target variable is `FuelAdjustedDegAbsolute`: cumulative seconds lost to tyre wear since the start of the stint, with the fuel load effect removed. A value of 1.2 means the tyre is currently 1.2s slower than it was on lap 1 of that stint — purely due to rubber degradation. Predicting this one step ahead (lap t+1 from laps 1..t) is the core task.

The model is built in PyTorch using inherited `nn.Module` blocks and wrapped in PyTorch Lightning for training. We go through two training phases: Phase 1 trains on 2023 and validates on 2024 — this is where we tune hyperparameters and run the production vs. pure features ablation. Phase 2 trains on 2023+2024 and tests on 2025 for the final numbers. The checkpoint from here feeds directly into N10, which fine-tunes one model per compound.

**Inputs:** `laps_tiredeg.parquet` · `tiredeg_sequence_config.json` · `tiredeg_feature_manifest.json`  
**Outputs:** `tiredeg_tcn_v1.ckpt` · `tiredeg_scaler.pkl` · `tiredeg_model_config.json`

---

| Step | |
|------|-|
| 0 | Imports and environment setup |
| 1 | Load data, sequence config and feature sets |
| 2 | Dataset, normalization and DataModule |
| 3 | Model architecture: `CausalConv1dBlock` → `TCNResidualBlock` → `TireDegTCN` |
| 4 | `TireDegLitModule`: loss, metrics and optimizer |
| 5 | Architecture profiling and GPU memory analysis |
| 6 | Phase 1a — Global model, production features |
| 7 | Phase 1b — Ablation with pure features |
| 8 | Phase 2 — Final model on 2023+2024, test on 2025 |
| 9 | Diagnostics: residuals by compound, cluster and tyre age |
| 10 | Save weights, scaler and config for N10 |
